# Step 10: DMRG with TeNPy

Exact diagonalisation (notebooks 05–07) is limited to $\sim 40$ sites. In notebooks 08–09 we saw that ground states of gapped systems have low entanglement and can be faithfully compressed into a Matrix Product State with a modest bond dimension $\chi$. What we have not yet done is find that MPS directly — without first computing the full $2^N$-dimensional state.

**DMRG** (Density Matrix Renormalisation Group, White 1992) does exactly this. It directly optimises an MPS ansatz by sweeping through the chain, solving a small eigenvalue problem at each site, and truncating back to bond dimension $\chi$. The key insight from notebook 09 — that fixing all tensors except one reduces the variational problem to an eigenvalue problem of size $O(d \chi^2)$ — is what makes this computationally tractable.

We use **TeNPy** (Tensor Network Python), a well-maintained library that implements DMRG and related algorithms, handles symmetries automatically, and provides a clean interface for defining models and extracting observables.

**What you will do:**
- Set up TeNPy models for the TFIM and Heisenberg chain
- Run DMRG and benchmark the results against our ED data
- Study convergence with bond dimension $\chi$ and the discarded weight
- Scan $\Gamma/J$ to map out the TFIM phase diagram at large $N$
- Push the Heisenberg chain to $N = 200$ and verify convergence to the Bethe ansatz

**Prerequisites:** Notebooks 06–09 (TFIM, ED, MPS).

## 10.1  The DMRG Algorithm

DMRG is a variational method: it minimises $\langle\psi|\hat{H}|\psi\rangle$ over all normalised MPS with bond dimension $\leq \chi$. The trick is to optimise one tensor at a time, keeping all others fixed.

**Local update.** Fix all tensors $\{A_i\}_{i \neq k}$. The energy as a function of $A_k$ alone is a Rayleigh quotient — its minimum is the ground state of the **effective Hamiltonian** $H_{\rm eff}$, a $d\chi^2 \times d\chi^2$ matrix built by contracting the full Hamiltonian with the fixed MPS tensors from the left and right. Solving this with Lanczos costs $O(\chi^3)$ and gives the optimal $A_k$ for the current environment.

**The sweep.** DMRG alternates between left-to-right and right-to-left sweeps, updating one (or two) tensors at each step. Each sweep lowers or preserves the energy. The algorithm terminates when the energy change per sweep falls below a threshold.

**Two-site DMRG.** In practice, updating two adjacent tensors simultaneously (two-site DMRG) gives better convergence because the SVD between them can grow the bond dimension naturally. After the local solve, an SVD truncates the two-site tensor back to bond dimension $\chi$, and the discarded weight $\epsilon = \sum_{\alpha > \chi} \lambda_\alpha^2$ is recorded as a convergence diagnostic.

The total cost per sweep is $O(N \chi^3 d^2)$ — linear in $N$ and polynomial in $\chi$, compared to the exponential cost of ED.

In [ ]:
# ── Check TeNPy installation ─────────────────────────────────────────────────
try:
    import tenpy
    print(f"TeNPy version: {tenpy.__version__}")
except ImportError:
    raise ImportError(
        "TeNPy is not installed. Install it with:\n"
        "    pip install physics-tenpy\n"
        "or\n"
        "    conda install -c conda-forge physics-tenpy"
    )

import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse.linalg import eigsh
from scipy.sparse import csr_matrix

import tenpy.linalg.np_conserved as npc
from tenpy.networks.mps import MPS
from tenpy.models.tf_ising import TFIChain
from tenpy.models.spins import SpinChain
from tenpy.algorithms import dmrg

# ── Reference ED implementation (from notebook 06) ───────────────────────────
def build_tfim_ed(N, J=1.0, Gamma=1.0, pbc=False):
    dim    = 2 ** N
    states = np.arange(dim)
    rows, cols, vals = [], [], []
    n_bonds = N if pbc else N - 1
    for i in range(n_bonds):
        j    = (i + 1) % N
        bi   = (states >> i) & 1
        bj   = (states >> j) & 1
        sz_i = 2 * bi.astype(float) - 1
        sz_j = 2 * bj.astype(float) - 1
        rows.extend(states.tolist()); cols.extend(states.tolist())
        vals.extend((-J * sz_i * sz_j).tolist())
    for i in range(N):
        s_flip = states ^ (1 << i)
        rows.extend(states.tolist()); cols.extend(s_flip.tolist())
        vals.extend([-Gamma] * dim)
    return csr_matrix((vals, (rows, cols)), shape=(dim, dim), dtype=float)

print("Imports successful.")

## 10.2  Defining Models in TeNPy

TeNPy provides built-in models for common Hamiltonians. The `TFIChain` model implements exactly our TFIM:

$$\hat{H} = -J \sum_i \hat{\sigma}^z_i \hat{\sigma}^z_{i+1} - g \sum_i \hat{\sigma}^x_i$$

with `g` playing the role of our $\Gamma$. The `SpinChain` model implements the Heisenberg chain:

$$\hat{H} = J \sum_i \hat{\vec{S}}_i \cdot \hat{\vec{S}}_{i+1}$$

with $J > 0$ corresponding to the antiferromagnet (opposite sign convention from notebook 05 where $J_{\rm AFM} = -1$).

A key TeNPy feature is automatic exploitation of conserved quantities. Setting `conserve='Sz'` for the Heisenberg chain restricts DMRG to the $S^z_{\rm tot} = 0$ sector, reducing the effective bond dimension by roughly a factor of $d$ and significantly speeding up the calculation.

In [ ]:
def make_tfim(N, J=1.0, Gamma=1.0):
    """Build TeNPy TFIChain model for given N, J, Gamma."""
    return TFIChain({
        'L'       : N,
        'J'       : J,
        'g'       : Gamma,      # transverse field = our Gamma
        'bc_MPS'  : 'finite',
        'conserve': 'None',     # TFIM has no U(1) symmetry
        'verbose' : 0,
    })


def make_heisenberg(N, J=1.0):
    """Build TeNPy SpinChain (Heisenberg) model.  J>0 is antiferromagnetic."""
    return SpinChain({
        'L'       : N,
        'Jx'      : J,
        'Jy'      : J,
        'Jz'      : J,
        'hz'      : 0.0,
        'bc_MPS'  : 'finite',
        'conserve': 'Sz',       # conserve total Sz — restricts to Sz_tot=0 sector
        'verbose' : 0,
    })


def run_dmrg(model, chi_max=32, mixer=True, max_sweeps=20,
             initial_state=None, verbose=False):
    """
    Run two-site DMRG for the given TeNPy model.

    Returns (E, psi, info) where:
      E    — ground state energy
      psi  — MPS ground state
      info — dict with convergence information
    """
    sites = model.lat.mps_sites()
    N     = model.lat.N_sites

    # Initialise MPS
    if initial_state is None:
        # Default: all-up for TFIM, Neel for Heisenberg
        conserve = model.lat.mps_sites()[0].conserve
        if conserve == 'Sz':
            # Neel state for Sz-conserving models
            product_state = ['up' if i % 2 == 0 else 'down' for i in range(N)]
        else:
            product_state = ['up'] * N
    else:
        product_state = initial_state

    psi0 = MPS.from_lat_product_state(model.lat, [product_state])

    # DMRG parameters
    dmrg_params = {
        'trunc_params': {
            'chi_max': chi_max,
            'svd_min': 1e-10,
        },
        'N_sweeps_check': 1,
        'max_sweeps'    : max_sweeps,
        'mixer'         : 'DensityMatrixMixer' if mixer else None,
        'mixer_params'  : {'amplitude': 1e-3, 'decay': 1.5, 'disable_after': 6},
        'verbose'       : 1 if verbose else 0,
    }

    eng  = dmrg.TwoSiteDMRGEngine(psi0, model, dmrg_params)
    E, psi = eng.run()
    info = eng.sweep_stats
    return E, psi, info


# Quick test: N=10 TFIM at Gamma/J = 1 (critical point)
N_test, J_test = 10, 1.0
model_test = make_tfim(N_test, J=J_test, Gamma=J_test)
E_dmrg, psi_dmrg, _ = run_dmrg(model_test, chi_max=16)

E_ed = eigsh(build_tfim_ed(N_test, J=J_test, Gamma=J_test),
             k=1, which='SA', return_eigenvectors=False)[0]

print(f"N={N_test}, Gamma/J=1 (critical):")
print(f"  DMRG energy:  {E_dmrg:.10f}")
print(f"  ED   energy:  {E_ed:.10f}")
print(f"  Difference:   {abs(E_dmrg - E_ed):.2e}")

## 10.3  Benchmarking DMRG Against ED

Before using DMRG for large systems, we validate it against our ED code for system sizes where both are feasible. This is standard practice: any new method must agree with ED for small systems before being trusted at large ones.

We compare ground state energies at three points in the TFIM phase diagram (ordered, critical, disordered) for several system sizes up to $N = 20$.

In [ ]:
J = 1.0
Gamma_benchmarks = [0.5, 1.0, 2.0]
label_benchmarks = ['Ordered ($\\Gamma/J=0.5$)', 'Critical ($\\Gamma=J$)', 'Disordered ($\\Gamma/J=2$)']
N_benchmarks     = [8, 10, 12, 14, 16, 18, 20]

print(f"{'N':>4}  {'Gamma/J':>8}  {'ED energy':>14}  {'DMRG energy':>14}  {'|diff|':>10}")
print("-" * 60)

benchmark_results = {}
for G, lbl in zip(Gamma_benchmarks, label_benchmarks):
    ed_energies   = []
    dmrg_energies = []
    for N in N_benchmarks:
        # ED
        e_ed = eigsh(build_tfim_ed(N, J=J, Gamma=G), k=1, which='SA',
                     return_eigenvectors=False)[0]
        # DMRG with chi large enough to be exact for these small systems
        model  = make_tfim(N, J=J, Gamma=G)
        e_dmrg, _, _ = run_dmrg(model, chi_max=64)

        ed_energies.append(e_ed)
        dmrg_energies.append(e_dmrg)
        print(f"{N:>4}  {G:>8.1f}  {e_ed:>14.8f}  {e_dmrg:>14.8f}  {abs(e_dmrg-e_ed):>10.2e}")

    benchmark_results[G] = (ed_energies, dmrg_energies)

print("\nAll differences should be < 1e-8 (limited by ED tolerance, not DMRG).")

## 10.4  Convergence with Bond Dimension $\chi$

DMRG accuracy is controlled by the bond dimension $\chi$. We track:

- The **energy error** $|E_\chi - E_{\rm exact}|/|E_{\rm exact}|$ as a function of $\chi$.
- The **discarded weight** $\epsilon = \sum_{\alpha > \chi} \lambda_\alpha^2$ at the central bond, reported by TeNPy after each sweep.

The discarded weight is the standard convergence diagnostic in DMRG: when it falls below $\sim 10^{-8}$, the bond dimension is sufficient and the energy is essentially exact. For a gapped system this happens at a small fixed $\chi$; at the critical point it requires increasingly large $\chi$.

In [ ]:
N_chi = 24
chi_values = [2, 4, 8, 16, 32, 64]

Gamma_chi_cases = [
    (0.4, 'Ordered ($\\Gamma/J=0.4$)',    '#2166ac'),
    (1.0, 'Critical ($\\Gamma=J$)',        '#d6604d'),
    (2.0, 'Disordered ($\\Gamma/J=2.0$)', '#1a9850'),
]

# Exact reference from ED
exact_energies = {}
for G, lbl, col in Gamma_chi_cases:
    exact_energies[G] = eigsh(build_tfim_ed(N_chi, J=J, Gamma=G), k=1, which='SA',
                              return_eigenvectors=False)[0]

chi_errors = {G: [] for G, _, _ in Gamma_chi_cases}

for G, lbl, col in Gamma_chi_cases:
    print(f"{lbl}...", flush=True)
    for chi in chi_values:
        model  = make_tfim(N_chi, J=J, Gamma=G)
        E_chi, _, _ = run_dmrg(model, chi_max=chi)
        err = abs(E_chi - exact_energies[G]) / abs(exact_energies[G])
        chi_errors[G].append(err)
        print(f"  chi = {chi:>3}: E = {E_chi:.8f}, rel. error = {err:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for G, lbl, col in Gamma_chi_cases:
    errs = np.array(chi_errors[G])
    axes[0].semilogy(chi_values, np.maximum(errs, 1e-16), 'o-', ms=6, lw=2,
                     color=col, label=lbl)
    axes[1].loglog(chi_values, np.maximum(errs, 1e-16), 'o-', ms=6, lw=2,
                   color=col, label=lbl)

for ax in axes:
    ax.set_xlabel(r'Bond dimension $\chi$', fontsize=13)
    ax.set_ylabel('Relative energy error', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3, which='both')
axes[0].set_title(f'Energy convergence vs $\\chi$  ($N={N_chi}$)', fontsize=13)
axes[1].set_title('Log-log: exponential vs power-law', fontsize=13)

plt.tight_layout()
plt.show()

## 10.5  Phase Diagram from DMRG

With a validated DMRG implementation, we now scan across $\Gamma/J$ at a system size $N = 100$ — far beyond the reach of ED. At each $\Gamma/J$ we extract:

- **Energy per site** $E_0/N$
- **Order parameter** $m = |\langle \hat{\sigma}^z \rangle|$ averaged over the bulk sites (excluding boundaries to reduce edge effects)
- **Entanglement entropy** at the central cut

These three observables cleanly distinguish the ordered, critical, and disordered phases.

In [ ]:
N_pd    = 100
chi_pd  = 64
Gamma_scan = np.concatenate([
    np.linspace(0.2, 0.8, 7),
    np.linspace(0.85, 1.15, 10),   # dense near critical point
    np.linspace(1.2, 2.0, 7),
])

E_per_site = []
order_param = []
entropy_center = []

# Bulk sites (exclude 10 sites from each boundary to avoid edge effects)
bulk_start = 10
bulk_end   = N_pd - 10

for G in Gamma_scan:
    print(f"Gamma/J = {G:.3f} ...", end=' ', flush=True)
    model = make_tfim(N_pd, J=J, Gamma=G)

    # In ordered phase, start from all-up to bias towards one ground state
    init = ['up'] * N_pd
    E, psi, _ = run_dmrg(model, chi_max=chi_pd, initial_state=init)

    # Energy per site
    E_per_site.append(E / N_pd)

    # Order parameter: |<Sigmaz>| averaged over bulk
    Sz_vals = psi.expectation_value('Sigmaz')
    m = np.mean(np.abs(Sz_vals[bulk_start:bulk_end]))
    order_param.append(m)

    # Entanglement entropy at central cut
    S_arr = psi.entanglement_entropy()
    entropy_center.append(S_arr[N_pd // 2 - 1])

    print(f"E/N={E/N_pd:.5f},  m={m:.4f},  S={entropy_center[-1]:.4f}")

E_per_site     = np.array(E_per_site)
order_param    = np.array(order_param)
entropy_center = np.array(entropy_center)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(Gamma_scan, E_per_site, 'o-', ms=5, lw=1.8, color='steelblue')
axes[0].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[0].set_xlabel(r'$\Gamma/J$', fontsize=13)
axes[0].set_ylabel(r'$E_0/N$', fontsize=13)
axes[0].set_title('Energy per site', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

axes[1].plot(Gamma_scan, order_param, 'o-', ms=5, lw=1.8, color='#d6604d')
axes[1].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[1].set_xlabel(r'$\Gamma/J$', fontsize=13)
axes[1].set_ylabel(r'$|\langle\hat{\sigma}^z\rangle|$ (bulk avg)', fontsize=13)
axes[1].set_title('Order parameter', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

axes[2].plot(Gamma_scan, entropy_center, 'o-', ms=5, lw=1.8, color='#1a9850')
axes[2].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[2].set_xlabel(r'$\Gamma/J$', fontsize=13)
axes[2].set_ylabel(r'$S_A$ (central cut)', fontsize=13)
axes[2].set_title('Entanglement entropy', fontsize=13)
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.3)

plt.suptitle(f'TFIM phase diagram from DMRG  ($N={N_pd}$, $\\chi={chi_pd}$)', fontsize=13)
plt.tight_layout()
plt.show()

## 10.6  Heisenberg Chain: Convergence to the Bethe Ansatz

The 1D antiferromagnetic Heisenberg chain has the exact result (Bethe ansatz):

$$\frac{E_0}{N} \xrightarrow{N\to\infty} \frac{1}{4} - \ln 2 \approx -0.4431$$

In notebook 05 we tracked this convergence up to $N = 20$ with ED. DMRG allows us to push to $N = 200$ and beyond, directly observing the $1/N^2$ finite-size correction from hundreds of sites rather than tens.

The Heisenberg chain benefits significantly from the `conserve='Sz'` symmetry: DMRG works only in the $S^z_{\rm tot} = 0$ sector, reducing the effective dimension by a factor of $d$ at each bond and allowing larger systems with the same computational cost.

In [ ]:
E_BETHE = 0.25 - np.log(2)   # exact: 1/4 - ln(2) ≈ -0.4431

# System sizes: ED (small) + DMRG (large)
N_ed_heisenberg   = [6, 8, 10, 12, 14, 16, 18, 20]
N_dmrg_heisenberg = [30, 50, 80, 100, 150, 200]
chi_heisenberg    = 64   # sufficient for gapped-like convergence

from scipy.sparse import csr_matrix

def build_heisenberg_ed(N, J=-1.0, pbc=True):
    dim = 2**N; states = np.arange(dim)
    rows, cols, vals = [], [], []
    bonds = [(i, (i+1)%N) for i in range(N if pbc else N-1)]
    for (i, j) in bonds:
        bi = (states>>i)&1; bj = (states>>j)&1
        sz_i = bi.astype(float)-0.5; sz_j = bj.astype(float)-0.5
        rows.extend(states.tolist()); cols.extend(states.tolist())
        vals.extend((-J*sz_i*sz_j).tolist())
        mask = bi!=bj; s_a = states[mask]; s_f = s_a^(1<<i)^(1<<j)
        rows.extend(s_f.tolist()); cols.extend(s_a.tolist())
        vals.extend([-J*0.5]*int(mask.sum()))
    return csr_matrix((vals,(rows,cols)),shape=(dim,dim),dtype=float)

# ED energies
print("ED (Heisenberg AFM, PBC):")
e0_ed_heis = []
for N in N_ed_heisenberg:
    H = build_heisenberg_ed(N, J=-1.0, pbc=True)
    e0 = eigsh(H, k=1, which='SA', return_eigenvectors=False)[0]
    e0_ed_heis.append(e0 / N)
    print(f"  N={N:>3}: E0/N = {e0/N:.8f}  (err = {abs(e0/N - E_BETHE):.2e})")

# DMRG energies
print("\nDMRG (Heisenberg AFM, OBC):")
e0_dmrg_heis = []
for N in N_dmrg_heisenberg:
    model = make_heisenberg(N, J=1.0)   # J=1 in TeNPy = AFM
    E, psi, _ = run_dmrg(model, chi_max=chi_heisenberg, mixer=True)
    e0_per_site = E / N
    e0_dmrg_heis.append(e0_per_site)
    print(f"  N={N:>3}: E0/N = {e0_per_site:.8f}  (err = {abs(e0_per_site - E_BETHE):.2e})")

print(f"\nBethe ansatz (N→∞): E0/N = {E_BETHE:.8f}")

In [ ]:
# Plot convergence to Bethe ansatz
all_N     = N_ed_heisenberg + N_dmrg_heisenberg
all_E     = e0_ed_heis + e0_dmrg_heis
all_err   = [abs(e - E_BETHE) for e in all_E]

n_ed   = len(N_ed_heisenberg)
n_dmrg = len(N_dmrg_heisenberg)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(N_ed_heisenberg, e0_ed_heis, 'o', ms=7, color='steelblue', label='ED (PBC)')
axes[0].plot(N_dmrg_heisenberg, e0_dmrg_heis, 's', ms=7, color='#d6604d', label=f'DMRG (OBC, $\\chi={chi_heisenberg}$)')
axes[0].axhline(E_BETHE, color='k', ls='--', lw=1.2, label=f'Bethe ansatz = {E_BETHE:.4f}')
axes[0].set_xlabel('System size $N$', fontsize=13)
axes[0].set_ylabel('$E_0/N$', fontsize=13)
axes[0].set_title('Ground state energy per site', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Log-log error vs 1/N^2
inv_N2    = [1/N**2 for N in all_N]
axes[1].loglog([1/N**2 for N in N_ed_heisenberg], all_err[:n_ed],
               'o', ms=7, color='steelblue', label='ED')
axes[1].loglog([1/N**2 for N in N_dmrg_heisenberg], all_err[n_ed:],
               's', ms=7, color='#d6604d', label='DMRG')
# Guide: 1/N^2 slope
x_fit = np.logspace(np.log10(min(inv_N2)*0.8), np.log10(max(inv_N2)*1.2), 50)
axes[1].loglog(x_fit, 0.5 * x_fit, 'k--', lw=1.2, label=r'$\propto 1/N^2$')
axes[1].set_xlabel(r'$1/N^2$', fontsize=13)
axes[1].set_ylabel(r'$|E_0/N - E_{\rm Bethe}|$', fontsize=13)
axes[1].set_title('Finite-size error', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

# Entanglement entropy profile for large Heisenberg chain
N_large  = N_dmrg_heisenberg[-1]   # use largest
model_lg = make_heisenberg(N_large, J=1.0)
_, psi_lg, _ = run_dmrg(model_lg, chi_max=chi_heisenberg)
S_profile    = psi_lg.entanglement_entropy()
ell_vals     = np.arange(1, N_large)

# Fit Calabrese-Cardy (OBC): S = (c/6) * log((2N/pi)*sin(pi*ell/N)) + k
from scipy.optimize import curve_fit
x_cc = np.log((2 * N_large / np.pi) * np.sin(np.pi * ell_vals / N_large))
def cc_obc(x, c, k): return (c / 6.0) * x + k
popt, _ = curve_fit(cc_obc, x_cc, S_profile)
c_fit   = popt[0]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ell_vals, S_profile, lw=1.5, color='steelblue', label='DMRG entanglement entropy')
ax.plot(ell_vals, cc_obc(x_cc, *popt), 'r--', lw=1.5,
        label=f'Calabrese-Cardy fit: $c = {c_fit:.3f}$  (expected $c=1$)')
ax.set_xlabel(r'Cut position $\ell$', fontsize=13)
ax.set_ylabel(r'$S_A(\ell)$', fontsize=13)
ax.set_title(f'Entanglement entropy — Heisenberg chain  ($N={N_large}$, OBC)', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Central charge from Calabrese-Cardy fit: c = {c_fit:.4f}  (expected 1.0000 for Heisenberg)")

## 10.7  Understanding TeNPy Output and Convergence

A DMRG run with `verbose=1` prints at each sweep:
- **Energy:** Should decrease monotonically (variational principle guarantees this).
- **Maximum discarded weight:** The largest $\epsilon = \sum_{\alpha > \chi} \lambda_\alpha^2$ over all bonds in that sweep. When this falls below $10^{-8}$, the bond dimension is sufficient.
- **Maximum $\chi$:** The actual bond dimension used (may be less than `chi_max` if the state has lower entanglement).

Below we run a single DMRG calculation with verbose output to illustrate the convergence behaviour.

In [ ]:
# Verbose DMRG run: N=40 Heisenberg chain, watch convergence sweep by sweep
N_verbose  = 40
chi_verbose = 32

model_v = make_heisenberg(N_verbose, J=1.0)
sites_v = model_v.lat.mps_sites()
product_state_v = ['up' if i%2==0 else 'down' for i in range(N_verbose)]
psi_v0  = MPS.from_lat_product_state(model_v.lat, [product_state_v])

dmrg_params_v = {
    'trunc_params': {'chi_max': chi_verbose, 'svd_min': 1e-10},
    'N_sweeps_check': 1,
    'max_sweeps': 15,
    'mixer': 'DensityMatrixMixer',
    'mixer_params': {'amplitude': 1e-3, 'decay': 1.5, 'disable_after': 6},
    'verbose': 1,
}

print(f"DMRG convergence for Heisenberg chain N={N_verbose}, chi={chi_verbose}:")
print(f"Bethe ansatz target: E/N = {E_BETHE:.8f}")
print()

eng_v = dmrg.TwoSiteDMRGEngine(psi_v0, model_v, dmrg_params_v)
E_v, psi_v = eng_v.run()

print(f"\nFinal: E/N = {E_v/N_verbose:.8f}  (error = {abs(E_v/N_verbose - E_BETHE):.2e})")

## 10.8  DMRG vs ED: Capabilities and Limitations

| Property | Exact Diagonalisation | DMRG ($\chi$ fixed) |
|---|---|---|
| Memory | $O(2^N)$ | $O(N\chi^2 d)$ |
| Time per ground state | $O(k \cdot 2^N)$ (Lanczos) | $O(N\chi^3 d^2)$ per sweep |
| Accuracy | Exact | Controlled by $\chi$ |
| Max system size | $\sim 40$ (with symmetry) | $\sim 10^3$–$10^4$ (gapped) |
| Phase diagram | Scan all parameters at once | One point at a time |
| Excited states | Full spectrum | Targeting required |
| Geometry | Any | 1D (or narrow 2D) |

**Where DMRG excels:** Ground states of 1D gapped systems, where modest $\chi$ suffices and the linear cost in $N$ allows chains of thousands of sites. DMRG is also excellent for quasi-1D systems (ladders, cylinders of width $W \lesssim 10$) and for computing correlation functions and entanglement spectra.

**Where DMRG struggles:** 2D systems, where the entanglement across a cut grows as $S \sim W$ with the system width, requiring $\chi \sim e^W$ — exponential in the circumference. Critical 1D systems also require large and growing $\chi$. For these situations, quantum Monte Carlo or variational Monte Carlo methods are often preferred.

**The key message:** ED and DMRG are complementary tools. ED provides exact benchmarks for small systems; DMRG accesses the thermodynamic limit for 1D systems. The workflow is always: validate DMRG against ED at small $N$, then use DMRG to extrapolate.

## Exercises

1. **TFIM at large $N$.** Run DMRG on the TFIM at $\Gamma/J = 1$ (critical) for $N = 50, 100, 200$ with $\chi = 64$. Extract the entanglement entropy profile and fit the Calabrese-Cardy formula (PBC or OBC as appropriate). Verify that the fitted central charge converges towards $c = 1/2$.

2. **Discarded weight as a convergence criterion.** For the $N = 100$ TFIM at criticality, run DMRG with $\chi = 16, 32, 64, 128$ and record the maximum discarded weight after convergence. Plot discarded weight and energy error vs $\chi$ on the same axes and verify that the discarded weight is a reliable proxy for the energy error.

3. **Spin-1 Heisenberg chain.** Change the SpinChain spin to $S = 1$ by setting `S=1` in the model parameters. Run DMRG for $N = 20, 50, 100$ and extract the spectral gap $\Delta = E_1 - E_0$ (using `ExcitedMPS` or a second DMRG run with orthogonality projection). Does the gap converge to the Haldane gap $\Delta \approx 0.41J$?

4. **Boundary effects.** In DMRG with OBC, the entanglement entropy profile is not uniform: edge sites have lower entropy than bulk sites. For the critical TFIM ($N=100$, $\chi=64$), plot the full entropy profile $S(\ell)$ vs $\ell$ and compare to the OBC Calabrese-Cardy formula $S(\ell) = \frac{c}{6}\log\!\left(\frac{2N}{\pi}\sin\frac{\pi\ell}{N}\right) + k$.